# Singer Population and Activity, 1992-2025

This notebook answers questions about the observed population of singers in `minutes_pre95.db`.

**Operational definition:** the database reliably exposes people who led songs, so this notebook treats a "singer" as a distinct `leader_id` appearing in `song_leader_joins`. These are observed singing participants/leaders, not a complete attendance roster.

**Activity definition:** a singer is active in the interval from their first observed year through their last observed year. If their first observed year is 1991 or 1992, the start is treated as undetermined/left-censored. If their last observed year is 2025, the end is treated as undetermined/right-censored.

In [ ]:
from pathlib import Path
import os
import sqlite3

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 40)
plt.style.use("seaborn-v0_8-whitegrid")

DB_PATH = Path("minutes_pre95.db")
START_YEAR = 1992
END_YEAR = 2025
CURRENT_START = 2023
CURRENT_END = 2025

assert DB_PATH.exists(), f"Could not find {DB_PATH.resolve()}"

## Load Lesson-Level Observations

Each row below is one parsed song-leading event, joined to the minute year and leader name.

In [ ]:
with sqlite3.connect(DB_PATH) as con:
    lessons = pd.read_sql_query(
        """
        SELECT
            slj.id AS song_leader_join_id,
            slj.lesson_id,
            slj.minutes_id,
            slj.leader_id,
            leaders.name AS leader_name,
            minutes.Year AS year,
            minutes.Name AS minutes_name,
            minutes.Location AS minutes_location
        FROM song_leader_joins AS slj
        JOIN leaders ON leaders.id = slj.leader_id
        JOIN minutes ON minutes.id = slj.minutes_id
        WHERE minutes.Year BETWEEN ? AND ?
        """,
        con,
        params=(START_YEAR, END_YEAR),
    )

years = np.arange(START_YEAR, END_YEAR + 1)
print(f"lesson rows: {len(lessons):,}")
print(f"distinct observed singers/leaders: {lessons['leader_id'].nunique():,}")
print(f"year range: {lessons['year'].min()}-{lessons['year'].max()}")
# lessons.head()

## How Did the Observed Singer Population Change?

This counts distinct observed singers/leaders by year. The rolling line smooths short-term changes.

In [ ]:
yearly_population = (
    lessons.groupby("year")
    .agg(
        observed_singers=("leader_id", "nunique"),
        lessons=("song_leader_join_id", "count"),
        minutes=("minutes_id", "nunique"),
    )
    .reindex(years)
    .rename_axis("year")
    .reset_index()
)
yearly_population["observed_singers_3yr_centered"] = yearly_population["observed_singers"].rolling(3, center=True, min_periods=1).mean()

# display(yearly_population)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(yearly_population["year"], yearly_population["observed_singers"], marker="o", linewidth=1, label="annual")
ax.plot(yearly_population["year"], yearly_population["observed_singers_3yr_centered"], linewidth=3, label="centered 3-year average")
ax.set_title("Observed Singer/Leader Population by Year")
ax.set_xlabel("Year")
ax.set_ylabel("Distinct observed singers/leaders")
ax.legend()
plt.show()

## How Many Leaders Were There Above Different Thresholds?

Thresholds are based on the number of lessons led in each period. Adjust `LESSON_THRESHOLDS` if another cutoff scheme is more useful.

In [ ]:
LESSON_THRESHOLDS = [1, 2, 5, 10, 25, 50, 100]
YEAR_THRESHOLDS = [1, 2, 3, 5, 10, 20, 30]

def leader_counts_for_period(start_year, end_year, label):
    period = lessons.query("@start_year <= year <= @end_year")
    by_leader = (
        period.groupby(["leader_id", "leader_name"])
        .agg(lesson_count=("song_leader_join_id", "count"), years_seen=("year", "nunique"))
        .reset_index()
    )
    rows = []
    for threshold in LESSON_THRESHOLDS:
        rows.append({"period": label, "basis": "lessons_led", "threshold_value": threshold, "threshold": f">= {threshold}", "leaders": int((by_leader["lesson_count"] >= threshold).sum())})
    for threshold in YEAR_THRESHOLDS:
        rows.append({"period": label, "basis": "years_seen", "threshold_value": threshold, "threshold": f">= {threshold}", "leaders": int((by_leader["years_seen"] >= threshold).sum())})
    return pd.DataFrame(rows), by_leader

period_specs = [
    (START_YEAR, END_YEAR, f"{START_YEAR}-{END_YEAR}"),
    (1992, 1994, "1992-1994"),
    (2023, 2025, "2023-2025"),
]

threshold_tables = []
period_leader_tables = {}
for start, end, label in period_specs:
    table, by_leader = leader_counts_for_period(start, end, label)
    threshold_tables.append(table)
    period_leader_tables[label] = by_leader

threshold_summary = pd.concat(threshold_tables, ignore_index=True)
threshold_order = pd.MultiIndex.from_tuples(
    [("lessons_led", threshold) for threshold in LESSON_THRESHOLDS]
    + [("years_seen", threshold) for threshold in YEAR_THRESHOLDS],
    names=["basis", "threshold_value"],
)
threshold_display = (
    threshold_summary.pivot_table(index=["basis", "threshold_value"], columns="period", values="leaders", aggfunc="first")
    .reindex(threshold_order)
    .reindex(columns=[f"{START_YEAR}-{END_YEAR}", "1992-1994", "2023-2025"])
)
threshold_display.index = pd.MultiIndex.from_arrays(
    [
        threshold_display.index.get_level_values("basis"),
        [f">= {threshold}" for threshold in threshold_display.index.get_level_values("threshold_value")],
    ],
    names=["basis", "threshold"],
)
threshold_display

## Rolling Three-Year Leader Counts

For each three-year window, this counts distinct leaders meeting the lesson thresholds inside that window.

In [ ]:
rolling_rows = []
for start in range(START_YEAR, END_YEAR - 1):
    end = start + 2
    window = lessons.query("@start <= year <= @end")
    by_leader = window.groupby("leader_id").size().rename("lesson_count")
    row = {"window_start": start, "window_end": end, "window": f"{start}-{end}", "distinct_leaders": int(by_leader.size)}
    for threshold in LESSON_THRESHOLDS:
        row[f"leaders_ge_{threshold}_lessons"] = int((by_leader >= threshold).sum())
    rolling_rows.append(row)

rolling_3yr = pd.DataFrame(rolling_rows)
# display(rolling_3yr)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(rolling_3yr["window_start"], rolling_3yr["distinct_leaders"], marker="o", label=">= 1 lesson")
for threshold in [2, 5, 10, 25, 50]:
    ax.plot(rolling_3yr["window_start"], rolling_3yr[f"leaders_ge_{threshold}_lessons"], marker="o", label=f">= {threshold} lessons")
ax.set_title("Leaders Above Lesson Thresholds in Rolling Three-Year Windows")
ax.set_xlabel("Window start year")
ax.set_ylabel("Distinct leaders")
ax.legend()
plt.show()

## Singer Activity Table

`observed_span_years` is `last_year - first_year + 1`. It is a lower bound whenever a singer is left- or right-censored.

In [ ]:
activity = (
    lessons.groupby(["leader_id", "leader_name"])
    .agg(
        first_year=("year", "min"),
        last_year=("year", "max"),
        years_seen=("year", "nunique"),
        lesson_count=("song_leader_join_id", "count"),
        minutes_count=("minutes_id", "nunique"),
    )
    .reset_index()
)

activity["observed_span_years"] = activity["last_year"] - activity["first_year"] + 1
activity["left_censored_start"] = activity["first_year"].isin([1991, 1992]) | (activity["first_year"] <= START_YEAR)
activity["right_censored_end"] = activity["last_year"] >= END_YEAR
activity["career_observed_complete"] = ~activity["left_censored_start"] & ~activity["right_censored_end"]
activity["active_2023_2025"] = (activity["first_year"] <= CURRENT_END) & (activity["last_year"] >= CURRENT_START)
activity = activity.sort_values(["lesson_count", "years_seen", "leader_name"], ascending=[False, False, True])

activity.head(20)

## How Many Leaders Were Active Across the Full Period?

The database covers 1992-2025. Calendar subtraction gives a 33-year endpoint-to-endpoint period; inclusive counting gives 34 observed calendar years. This section counts singers observed at both endpoints and singers observed in every calendar year.

In [ ]:
full_year_count = len(years)
active_across_endpoints = activity.query("first_year <= @START_YEAR and last_year >= @END_YEAR").copy()
seen_every_year = activity.query("years_seen == @full_year_count").copy()

full_period_summary = pd.DataFrame(
    [
        {"definition": "observed at both 1992 and 2025 endpoints", "leaders": len(active_across_endpoints)},
        {"definition": "observed in every calendar year, 1992-2025", "leaders": len(seen_every_year)},
    ]
)
display(full_period_summary)
display(active_across_endpoints[["leader_name", "first_year", "last_year", "years_seen", "lesson_count"]].sort_values(["years_seen", "lesson_count"], ascending=False))
display(seen_every_year[["leader_name", "lesson_count"]].sort_values(["lesson_count"], ascending=False))

## How Long Are Singers Typically Active?

Use `observed_span_years` as a lower bound for censored singers. The cleanest estimate of completed careers excludes singers whose start or end is censored.

The filtered table asks the same question among leaders who led more than `N` lessons during the full observation period.

In [ ]:
def summarize_series(series):
    return pd.Series(
        {
            "count": int(series.count()),
            "mean": series.mean(),
            "median": series.median(),
            "p25": series.quantile(0.25),
            "p75": series.quantile(0.75),
            "min": series.min(),
            "max": series.max(),
        }
    )

career_summary = pd.DataFrame(
    {
        "all_observed_lower_bound": summarize_series(activity["observed_span_years"]),
        "completed_observed_careers_only": summarize_series(activity.loc[activity["career_observed_complete"], "observed_span_years"]),
        "left_censored_start": summarize_series(activity.loc[activity["left_censored_start"], "observed_span_years"]),
        "right_censored_end": summarize_series(activity.loc[activity["right_censored_end"], "observed_span_years"]),
    }
).T
display(career_summary)

ACTIVE_LESSON_COUNT_THRESHOLDS = [0, 1, 2, 5, 10, 25, 50, 100]

lesson_count_filtered_rows = []
for threshold in ACTIVE_LESSON_COUNT_THRESHOLDS:
    filtered = activity.query("lesson_count > @threshold")
    completed = filtered.query("career_observed_complete")
    for label, df in [
        ("all_observed_lower_bound", filtered),
        ("completed_observed_careers_only", completed),
    ]:
        summary = summarize_series(df["observed_span_years"])
        lesson_count_filtered_rows.append(
            {
                "lesson_count_filter": f"> {threshold}",
                "population": label,
                **summary.to_dict(),
            }
        )

lesson_count_filtered_activity = pd.DataFrame(lesson_count_filtered_rows)
lesson_count_filtered_activity

## How Long Have Today's Singers Been Active?

"Today's singers" means anyone observed during 2023-2025.

In [ ]:
current_singers = activity.query("active_2023_2025").copy()

current_summary = pd.DataFrame(
    {
        "current_singers_observed_span_lower_bound": summarize_series(current_singers["observed_span_years"]),
        "current_singers_years_seen": summarize_series(current_singers["years_seen"]),
    }
).T
display(current_summary)

current_singers[["leader_name", "first_year", "last_year", "observed_span_years", "years_seen", "lesson_count", "left_censored_start", "right_censored_end"]].sort_values(
    ["observed_span_years", "lesson_count"], ascending=False
).head(50)

## Percent Active for 1-5, 6-10, 11-15, 16-20, etc. Years

These buckets use observed active span, not number of years actually seen. A singer first observed in 2000 and last observed in 2010 has an 11-year observed span even if they only led in some of those years.

In [ ]:
bucket_edges = [1, 6, 11, 16, 21, 26, 31, np.inf]
bucket_labels = ["1-5", "6-10", "11-15", "16-20", "21-25", "26-30", "31+"]

def bucket_table(df, label):
    buckets = pd.cut(df["observed_span_years"], bins=bucket_edges, labels=bucket_labels, right=False)
    out = buckets.value_counts(sort=False).rename_axis("observed_active_span_years").reset_index(name="singers")
    out["percent"] = out["singers"] / out["singers"].sum() * 100
    out["population"] = label
    return out[["population", "observed_active_span_years", "singers", "percent"]]

span_buckets = pd.concat(
    [
        bucket_table(activity, "all observed singers"),
        bucket_table(activity.query("career_observed_complete"), "completed observed careers only"),
        bucket_table(current_singers, "active in 2023-2025"),
    ],
    ignore_index=True,
)
display(span_buckets)

plot_data = span_buckets.pivot(index="observed_active_span_years", columns="population", values="percent")
ax = plot_data.plot(kind="bar", figsize=(11, 5))
ax.set_title("Observed Active Span Distribution")
ax.set_xlabel("Observed active span in years")
ax.set_ylabel("Percent of singers")
plt.xticks(rotation=0)
plt.show()

## Average Singing Career During This Period

Because some singers started before the observation window and some are still active at the end, there is no single unbiased average without a survival model or external pre/post data. This notebook reports three useful quantities:

1. All observed spans: a lower-bound average.
2. Completed observed careers only: excludes left- and right-censored singers.
3. Current singers: a lower-bound tenure for people active in 2023-2025.

In [ ]:
average_career_answers = pd.DataFrame(
    [
        {
            "measure": "All observed singers, lower-bound span",
            "singers": len(activity),
            "mean_years": activity["observed_span_years"].mean(),
            "median_years": activity["observed_span_years"].median(),
            "note": "Includes left/right censored singers; underestimates true careers for censored people.",
        },
        {
            "measure": "Completed observed careers only",
            "singers": int(activity["career_observed_complete"].sum()),
            "mean_years": activity.loc[activity["career_observed_complete"], "observed_span_years"].mean(),
            "median_years": activity.loc[activity["career_observed_complete"], "observed_span_years"].median(),
            "note": "Less censored, but biased toward people whose full observed career fits inside 1992-2025.",
        },
        {
            "measure": "Active in 2023-2025, lower-bound span",
            "singers": len(current_singers),
            "mean_years": current_singers["observed_span_years"].mean(),
            "median_years": current_singers["observed_span_years"].median(),
            "note": "Tenure so far for today's observed singers; many are right-censored.",
        },
    ]
)
average_career_answers

## Export Tables for Reuse

Uncomment this cell if you want CSV snapshots of the main derived tables.

In [ ]:
# out_dir = Path("analysis_outputs")
# out_dir.mkdir(exist_ok=True)
# activity.to_csv(out_dir / "singer_activity_1992_2025.csv", index=False)
# yearly_population.to_csv(out_dir / "yearly_observed_singer_population.csv", index=False)
# threshold_summary.to_csv(out_dir / "leader_threshold_summary.csv", index=False)
# rolling_3yr.to_csv(out_dir / "rolling_3yr_leader_thresholds.csv", index=False)
# span_buckets.to_csv(out_dir / "observed_active_span_buckets.csv", index=False)